# 🔍 FastReID — Person Re-Identification on Market1501
> Based on: *FastReID: A Pytorch Toolbox for General Instance Re-identification* (He et al., 2020)

**Best config from paper:**
- Backbone: IBN-ResNet101 + Non-local module
- Aggregation: GeM Pooling
- Head: BNNeck
- Loss: Circle Loss + Triplet Loss
- Post-processing: Query Expansion (QE) + Re-ranking
- Target: **Rank-1 = 96.3%, mAP = 90.3%** on Market1501


## 📦 Step 1 — Environment Setup

In [ ]:
import subprocess, sys

def run(cmd):
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if result.returncode != 0:
        print(result.stderr[-2000:])
    else:
        print(result.stdout[-500:] or '✅ Done')

# Clone FastReID if not already present
import os
if not os.path.exists('fast-reid'):
    run('git clone https://github.com/JDAI-CV/fast-reid.git')
os.chdir('fast-reid')

# Install dependencies
run('pip install -q torch torchvision --extra-index-url https://download.pytorch.org/whl/cu118')
run('pip install -q faiss-gpu timm yacs termcolor tabulate tensorboard')
run('pip install -q opencv-python-headless scikit-learn matplotlib seaborn tqdm ipywidgets')

# Install FastReID
run('pip install -q -e .')

print('\n✅ Environment ready!')

In [ ]:
import torch
import platform

print('='*50)
print(f'Python     : {sys.version.split()[0]}')
print(f'PyTorch    : {torch.__version__}')
print(f'CUDA       : {torch.version.cuda}')
print(f'GPU count  : {torch.cuda.device_count()}')
for i in range(torch.cuda.device_count()):
    name = torch.cuda.get_device_name(i)
    mem  = torch.cuda.get_device_properties(i).total_memory / 1e9
    print(f'  GPU {i}    : {name}  ({mem:.1f} GB VRAM)')
print('='*50)

## 📁 Step 2 — Download Market1501 Dataset

In [ ]:
import os, zipfile, urllib.request
from pathlib import Path

# ─── CONFIG ───────────────────────────────────────────────────────────────────
DATASET_ROOT = Path('../datasets')          # where Market-1501 will live
MARKET_URL   = 'https://drive.usercontent.google.com/download?id=0B8-rUzbwVRk0c054eEozWG9COHM&export=download&confirm=t'
# NOTE: Google Drive direct download may require gdown
# ──────────────────────────────────────────────────────────────────────────────

DATASET_ROOT.mkdir(parents=True, exist_ok=True)
market_dir = DATASET_ROOT / 'Market-1501-v15.09.15'

if market_dir.exists():
    print(f'✅ Dataset already at: {market_dir}')
else:
    print('Downloading Market1501 via gdown ...')
    run('pip install -q gdown')
    run(f'gdown --id 0B8-rUzbwVRk0c054eEozWG9COHM -O {DATASET_ROOT}/Market-1501.zip')
    print('Extracting...')
    with zipfile.ZipFile(f'{DATASET_ROOT}/Market-1501.zip') as z:
        z.extractall(str(DATASET_ROOT))
    print(f'✅ Extracted to {market_dir}')

# Verify structure
for sub in ['bounding_box_train', 'bounding_box_test', 'query']:
    path = market_dir / sub
    count = len(list(path.glob('*.jpg'))) if path.exists() else 0
    print(f'  {sub:25s}: {count:6d} images')

## ⚙️ Step 3 — Generate Config YAML (Best Paper Settings)

In [ ]:
import yaml
from pathlib import Path

CONFIG_PATH = Path('configs/Market1501/best_config.yml')
CONFIG_PATH.parent.mkdir(parents=True, exist_ok=True)

config = """
MODEL:
  META_ARCHITECTURE: Baseline
  BACKBONE:
    NAME: build_resnet_backbone
    DEPTH: 101x           # IBN-ResNet101
    LAST_STRIDE: 1
    WITH_IBN: True
    WITH_NL: True          # Non-local module
    PRETRAIN: True
    PRETRAIN_PATH: ''
  HEADS:
    NAME: EmbeddingHead
    POOL_LAYER: gempooling  # GeM Pooling (paper Section 3.3)
    NECK: bnneck            # BNNeck head
    NECK_FEAT: before
    CLS_LAYER: linear
    NORM: False
    IN_FEAT: 2048
    NUM_CLASSES: 751        # Market1501 identities
  LOSSES:
    NAME: ('CrossEntropyLoss', 'TripletLoss', 'CircleLoss')
    CE:
      EPSILON: 0.1          # Label smoothing
      SCALE: 1.0
    TRI:
      MARGIN: 0.3
      HARD_MINING: True
      SCALE: 1.0
    CIRCLE:
      SCALE: 128
      MARGIN: 0.25

INPUT:
  SIZE_TRAIN: [384, 128]
  SIZE_TEST:  [384, 128]
  PROB: 0.5
  PADDING: 10
  DO_AUGMIX: False
  DO_AUTOAUG: True          # Auto-augment
  DO_FLIP: True
  RE:
    ENABLED: True           # Random erasing
    PROB: 0.5
  DO_RANDOMPATCH: True      # Random patch

DATASETS:
  NAMES: ('Market1501',)
  ROOT_DIR: '../datasets'

DATALOADER:
  NUM_WORKERS: 8
  SAMPLER: RandomIdentitySampler
  NUM_INSTANCE: 16          # 16 images per identity

SOLVER:
  OPT: Adam
  MAX_ITER: 18000
  BASE_LR: 0.00035          # 3.5e-4 (paper Fig. 4)
  BIAS_LR_FACTOR: 1.0
  MOMENTUM: 0.9
  WEIGHT_DECAY: 0.0005
  WEIGHT_DECAY_BIAS: 0.0005
  DELAY_ITERS: 9000         # Cosine decay starts at 9k
  IMS_PER_BATCH: 64         # 4 subjects × 16 images
  CHECKPOINT_PERIOD: 2000
  LOG_PERIOD: 100
  WARMUP_FACTOR: 0.01
  WARMUP_ITERS: 2000        # Warmup for 2k iters (paper)
  WARMUP_METHOD: linear
  FREEZE_ITERS: 2000        # Backbone freeze for 2k iters
  COSINE_MAIN_LR: 3.5e-4
  COSINE_END_LR: 7.7e-7     # Cosine decay end (paper)
  ETA_MIN_LR: 7.7e-7

TEST:
  EVAL_PERIOD: 2000
  IMS_PER_BATCH: 256
  METRIC: cosine
  ROC_ENABLED: True
  RERANK:
    ENABLED: False           # Enable for final eval
    K1: 20
    K2: 6
    LAMBDA: 0.3
  AQE:
    ENABLED: False           # Query Expansion - final eval
    ALPHA: 3.0
    QE_TIME: 1
    QE_K: 5

OUTPUT_DIR: '../outputs/market1501_best'
"""

CONFIG_PATH.write_text(config)
print(f'✅ Config written to: {CONFIG_PATH}')

## 🚀 Step 4 — Training

In [ ]:
import subprocess, threading, time
from IPython.display import clear_output

NUM_GPUS = torch.cuda.device_count()
print(f'Training on {NUM_GPUS} GPU(s)...')

train_cmd = (
    f'python tools/train_net.py '
    f'--config-file configs/Market1501/best_config.yml '
    f'--num-gpus {NUM_GPUS} '
    f'2>&1 | tee ../outputs/train_log.txt'
)

print('Command:', train_cmd)
print('\n▶ Starting training... (monitor ../outputs/train_log.txt)')
print('  This will take 4–8 hours on a single V100/A100.')
print('  Run the MONITORING cell below in a separate cell while this runs.')

# Uncomment to actually run:
# !{train_cmd}

In [ ]:
# ─── LIVE LOG MONITOR (run in parallel cell) ───────────────────────────────
# Run this while training is happening to see live output

LOG_FILE = '../outputs/train_log.txt'

def tail_log(n_lines=30):
    if not os.path.exists(LOG_FILE):
        print('Log not started yet...')
        return
    with open(LOG_FILE) as f:
        lines = f.readlines()
    for line in lines[-n_lines:]:
        print(line, end='')

tail_log()

## 📊 Step 5 — Parse Training Log & Plot Metrics

In [ ]:
import re, json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
from matplotlib.ticker import MaxNLocator
import seaborn as sns

# ─── STYLE ────────────────────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.facecolor': '#0d1117',
    'axes.facecolor':   '#161b22',
    'axes.edgecolor':   '#30363d',
    'text.color':       '#e6edf3',
    'axes.labelcolor':  '#e6edf3',
    'xtick.color':      '#8b949e',
    'ytick.color':      '#8b949e',
    'grid.color':       '#21262d',
    'grid.linestyle':   '--',
    'font.family':      'monospace',
    'axes.titlesize':   13,
    'axes.labelsize':   11,
})

NEON_GREEN  = '#39d353'
NEON_BLUE   = '#58a6ff'
NEON_ORANGE = '#ff7b72'
NEON_PURPLE = '#bc8cff'
NEON_YELLOW = '#e3b341'

# ─── PARSE LOG ────────────────────────────────────────────────────────────────
def parse_fastreid_log(log_path):
    """
    Parse FastReID training log for:
    - Iteration, loss, learning rate
    - Validation Rank-1, mAP at checkpoints
    """
    iters, losses, lrs = [], [], []
    val_iters, rank1s, maps = [], [], []

    if not os.path.exists(log_path):
        print(f'⚠ Log not found: {log_path}  — using synthetic demo data.')
        return _synthetic_data()

    with open(log_path) as f:
        for line in f:
            # Training step
            m = re.search(r'iter:\s*(\d+).*?total_loss:\s*([\d.]+).*?lr:\s*([\d.e+-]+)', line)
            if m:
                iters.append(int(m.group(1)))
                losses.append(float(m.group(2)))
                lrs.append(float(m.group(3)))
            # Validation results
            m2 = re.search(r'Rank-1:\s*([\d.]+).*?mAP:\s*([\d.]+)', line)
            if m2 and iters:
                val_iters.append(iters[-1])
                rank1s.append(float(m2.group(1)))
                maps.append(float(m2.group(2)))

    if not iters:
        print('⚠ No training data parsed — using synthetic demo data.')
        return _synthetic_data()

    return {
        'iters': np.array(iters),
        'losses': np.array(losses),
        'lrs': np.array(lrs),
        'val_iters': np.array(val_iters),
        'rank1s': np.array(rank1s),
        'maps': np.array(maps),
    }


def _synthetic_data():
    """Simulate realistic FastReID Market1501 training curves (paper Fig. 4 + Table 1)."""
    iters = np.arange(100, 18001, 100)
    n = len(iters)

    # Learning rate: warmup → flat → cosine decay (matching paper Fig. 4)
    lrs = np.where(iters <= 2000,
                   3.5e-5 + (3.5e-4 - 3.5e-5) * (iters / 2000),
                   np.where(iters <= 9000,
                            3.5e-4,
                            3.5e-4 * 0.5 * (1 + np.cos(np.pi * (iters - 9000) / 9000)) + 7.7e-7))

    # Loss: high at start, drops after warmup, smooth decay
    base_loss = 6.5 * np.exp(-iters / 3500) + 0.65 + np.random.normal(0, 0.04, n)
    losses = np.clip(base_loss, 0.5, 7.5)

    # Validation checkpoints every 2000 iters
    val_iters = np.arange(2000, 18001, 2000)
    # Rank-1 climbing toward ~96.3% (paper result)
    rank1s = np.array([72.1, 82.4, 88.7, 91.2, 93.5, 94.8, 95.5, 96.3, 96.3])
    maps   = np.array([51.3, 67.8, 76.4, 81.5, 85.2, 87.8, 89.1, 90.3, 90.3])
    rank1s += np.random.normal(0, 0.2, len(rank1s))
    maps   += np.random.normal(0, 0.2, len(maps))

    return {
        'iters': iters,
        'losses': losses,
        'lrs': lrs,
        'val_iters': val_iters,
        'rank1s': rank1s,
        'maps': maps,
        'synthetic': True,
    }


data = parse_fastreid_log('../outputs/train_log.txt')
print('✅ Parsed log data:')
for k, v in data.items():
    if isinstance(v, np.ndarray):
        print(f'  {k:12s}: {len(v)} points  (range: {v.min():.4f} – {v.max():.4f})')

In [ ]:
# ─── MAIN DASHBOARD PLOT ──────────────────────────────────────────────────────

fig = plt.figure(figsize=(18, 12), constrained_layout=True)
fig.patch.set_facecolor('#0d1117')

gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.35, wspace=0.3)

ax_lr   = fig.add_subplot(gs[0, 0])
ax_loss = fig.add_subplot(gs[0, 1])
ax_r1   = fig.add_subplot(gs[0, 2])
ax_map  = fig.add_subplot(gs[1, 0])
ax_comp = fig.add_subplot(gs[1, 1])
ax_bar  = fig.add_subplot(gs[1, 2])

iters = data['iters']
losses = data['losses']
lrs = data['lrs']
val_iters = data['val_iters']
rank1s = data['rank1s']
maps   = data['maps']

def style_ax(ax, title, xlabel='Iteration', ylabel=''):
    ax.set_title(title, color='#e6edf3', fontweight='bold', pad=8)
    ax.set_xlabel(xlabel, labelpad=6)
    ax.set_ylabel(ylabel, labelpad=6)
    ax.grid(True, alpha=0.3)
    ax.spines[['top','right']].set_visible(False)

# ── 1. Learning Rate ─────────────────────────────────────────────────────────
ax_lr.fill_between(iters, lrs, alpha=0.15, color=NEON_BLUE)
ax_lr.plot(iters, lrs, color=NEON_BLUE, lw=2)
ax_lr.axvline(2000, color=NEON_YELLOW, ls='--', lw=1.2, alpha=0.8, label='Warmup end')
ax_lr.axvline(9000, color=NEON_ORANGE, ls='--', lw=1.2, alpha=0.8, label='Cosine start')
ax_lr.set_yscale('log')
ax_lr.legend(fontsize=8, facecolor='#21262d', edgecolor='#30363d', labelcolor='#e6edf3')
style_ax(ax_lr, '📈 Learning Rate Schedule', ylabel='LR (log scale)')

# Add region labels
ax_lr.text(1000, lrs.max()*0.6, 'Warmup', ha='center', color=NEON_YELLOW, fontsize=7)
ax_lr.text(5500, lrs.max()*0.6, 'Plateau', ha='center', color='#8b949e', fontsize=7)
ax_lr.text(13500, lrs.max()*0.2, 'Cosine\nDecay', ha='center', color=NEON_ORANGE, fontsize=7)

# ── 2. Training Loss ──────────────────────────────────────────────────────────
# Smooth for display
from scipy.ndimage import uniform_filter1d
loss_smooth = uniform_filter1d(losses, size=15)
ax_loss.fill_between(iters, losses, alpha=0.08, color=NEON_ORANGE)
ax_loss.plot(iters, losses, color=NEON_ORANGE, lw=0.6, alpha=0.3)
ax_loss.plot(iters, loss_smooth, color=NEON_ORANGE, lw=2.2, label='Smoothed loss')
ax_loss.axvline(2000, color=NEON_YELLOW, ls='--', lw=1, alpha=0.6)
style_ax(ax_loss, '📉 Total Training Loss', ylabel='Loss')

# ── 3. Rank-1 Accuracy ───────────────────────────────────────────────────────
ax_r1.fill_between(val_iters, rank1s, alpha=0.15, color=NEON_GREEN)
ax_r1.plot(val_iters, rank1s, 'o-', color=NEON_GREEN, lw=2, ms=6, label='Rank-1')
ax_r1.axhline(96.3, color=NEON_GREEN, ls=':', lw=1.2, alpha=0.6, label='Paper target 96.3%')
for x, y in zip(val_iters, rank1s):
    ax_r1.annotate(f'{y:.1f}', (x, y), textcoords='offset points', xytext=(0,6),
                   color=NEON_GREEN, fontsize=7, ha='center')
ax_r1.set_ylim(60, 100)
ax_r1.legend(fontsize=8, facecolor='#21262d', edgecolor='#30363d', labelcolor='#e6edf3')
style_ax(ax_r1, '🎯 Rank-1 Accuracy (Market1501)', ylabel='Rank-1 (%)')

# ── 4. mAP ───────────────────────────────────────────────────────────────────
ax_map.fill_between(val_iters, maps, alpha=0.15, color=NEON_PURPLE)
ax_map.plot(val_iters, maps, 's-', color=NEON_PURPLE, lw=2, ms=6, label='mAP')
ax_map.axhline(90.3, color=NEON_PURPLE, ls=':', lw=1.2, alpha=0.6, label='Paper target 90.3%')
for x, y in zip(val_iters, maps):
    ax_map.annotate(f'{y:.1f}', (x, y), textcoords='offset points', xytext=(0,6),
                    color=NEON_PURPLE, fontsize=7, ha='center')
ax_map.set_ylim(40, 100)
ax_map.legend(fontsize=8, facecolor='#21262d', edgecolor='#30363d', labelcolor='#e6edf3')
style_ax(ax_map, '📐 Mean Average Precision (mAP)', ylabel='mAP (%)')

# ── 5. Combined R1 + mAP ─────────────────────────────────────────────────────
ax_comp.plot(val_iters, rank1s, 'o-', color=NEON_GREEN,  lw=2, ms=5, label='Rank-1')
ax_comp.plot(val_iters, maps,   's-', color=NEON_PURPLE, lw=2, ms=5, label='mAP')
ax_comp.fill_between(val_iters, rank1s, maps, alpha=0.07, color='white', label='Gap')
ax_comp.set_ylim(40, 100)
ax_comp.legend(fontsize=9, facecolor='#21262d', edgecolor='#30363d', labelcolor='#e6edf3')
style_ax(ax_comp, '📊 Rank-1 vs mAP Progress', ylabel='Score (%)')

# ── 6. Model Comparison Bar Chart (Table 1 from paper) ───────────────────────
methods = ['PCB\n(ECCV18)', 'AANet\n(CVPR19)', 'ABDNet\n(ICCV19)', 'SCAL\n(ICCV19)',
           'Circle\n(CVPR20)', 'FastReID\nR50', 'FastReID\nR50-IBN', 'FastReID\nR101-IBN']
r1_scores = [92.3, 93.9, 95.6, 95.8, 96.1, 95.4, 95.7, 96.3]
map_scores = [77.4, 83.4, 88.3, 89.3, 87.4, 88.2, 89.3, 90.3]

x = np.arange(len(methods))
w = 0.38
bars1 = ax_bar.bar(x - w/2, r1_scores, w, color=NEON_GREEN,  alpha=0.85, label='Rank-1')
bars2 = ax_bar.bar(x + w/2, map_scores, w, color=NEON_PURPLE, alpha=0.85, label='mAP')

# Highlight FastReID bars
for i, b in enumerate(bars1):
    if i >= 5:
        b.set_edgecolor('white')
        b.set_linewidth(1.5)
for i, b in enumerate(bars2):
    if i >= 5:
        b.set_edgecolor('white')
        b.set_linewidth(1.5)

ax_bar.set_xticks(x)
ax_bar.set_xticklabels(methods, fontsize=7)
ax_bar.set_ylim(70, 100)
ax_bar.legend(fontsize=8, facecolor='#21262d', edgecolor='#30363d', labelcolor='#e6edf3')
style_ax(ax_bar, '🏆 SOTA Comparison — Market1501', xlabel='Method', ylabel='Score (%)')
ax_bar.axvline(4.5, color='#30363d', lw=1.5, ls='--')  # separator
ax_bar.text(6.5, 71, 'FastReID', color='white', fontsize=7, ha='center',
            bbox=dict(boxstyle='round', fc='#21262d', ec='#30363d', alpha=0.8))

# ─── TITLE ───────────────────────────────────────────────────────────────────
fig.suptitle('FastReID — Market1501 Training Dashboard', 
             fontsize=16, color='white', fontweight='bold', y=1.01)

label = '⚠ Synthetic demo data' if data.get('synthetic') else '✅ Real training data'
fig.text(0.5, -0.01, label, ha='center', color='#8b949e', fontsize=9)

plt.savefig('../outputs/training_dashboard.png', dpi=150, bbox_inches='tight',
            facecolor=fig.get_facecolor())
plt.show()
print('✅ Saved → ../outputs/training_dashboard.png')

## 🧪 Step 6 — Re-ID Validation Viewer
> Inspired by your fall-detection UI — adapted for Re-ID query/gallery visualization

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import matplotlib.gridspec as gridspec
from matplotlib.patches import FancyBboxPatch
import numpy as np
import random

# ─── SYNTHETIC RANK LIST VISUALIZATION ────────────────────────────────────────
# In production, replace with actual query/gallery embeddings from FastReID

def make_person_placeholder(ax, color='#39d353', is_query=False):
    """Draw a stick-figure person like the skeleton in your reference image."""
    ax.set_facecolor('#161b22')
    ax.set_xlim(0, 10)
    ax.set_ylim(0, 20)
    ax.set_aspect('equal')
    ax.axis('off')

    lw = 2.5
    # Head
    head = plt.Circle((5, 18), 1.2, color=color, fill=True, zorder=3)
    ax.add_patch(head)
    # Torso
    ax.plot([5,5], [17,13], color=color, lw=lw, zorder=2)
    # Arms
    ax.plot([5,2.5], [16,14], color=color, lw=lw, zorder=2)
    ax.plot([5,7.5], [16,14], color=color, lw=lw, zorder=2)
    ax.plot([2.5,2], [14,11], color=color, lw=lw, zorder=2)
    ax.plot([7.5,8], [14,11], color=color, lw=lw, zorder=2)
    # Legs
    ax.plot([5,3.5], [13,9], color=color, lw=lw, zorder=2)
    ax.plot([5,6.5], [13,9], color=color, lw=lw, zorder=2)
    ax.plot([3.5,3], [9,5], color=color, lw=lw, zorder=2)
    ax.plot([6.5,7], [9,5], color=color, lw=lw, zorder=2)
    # Joints (red dots like your reference image)
    joints = [(5,18),(5,17),(5,15),(5,13),(2.5,16),(7.5,16),
               (2,11),(8,11),(3.5,9),(6.5,9),(3,5),(7,5)]
    for jx, jy in joints:
        dot = plt.Circle((jx,jy), 0.35, color='#ff4444', zorder=4)
        ax.add_patch(dot)

    if is_query:
        border = FancyBboxPatch((0.2,0.2), 9.6, 19.6, 
                                 boxstyle='round,pad=0.1',
                                 linewidth=3, edgecolor='#ffd700', facecolor='none')
        ax.add_patch(border)
        ax.text(5, 1, 'QUERY', ha='center', color='#ffd700', fontsize=7, fontweight='bold')


def reid_validation_panel(query_id=1, results=None, current_iter=18000,
                           rank1=96.3, map_score=90.3):
    """
    Draw a Re-ID validation panel similar to the fall-detection UI in the reference image.
    Shows: query image, top-K retrieved gallery images with match scores.
    """
    if results is None:
        # Simulated retrieval results [person_id, score, is_correct_match]
        np.random.seed(query_id)
        results = [
            {'id': query_id,  'score': 0.97, 'match': True},
            {'id': query_id,  'score': 0.94, 'match': True},
            {'id': query_id,  'score': 0.91, 'match': True},
            {'id': query_id+5,'score': 0.45, 'match': False},
            {'id': query_id,  'score': 0.88, 'match': True},
            {'id': query_id+3,'score': 0.38, 'match': False},
            {'id': query_id,  'score': 0.82, 'match': True},
            {'id': query_id+8,'score': 0.29, 'match': False},
            {'id': query_id,  'score': 0.78, 'match': True},
            {'id': query_id+2,'score': 0.21, 'match': False},
        ]

    TOP_K = len(results)
    n_correct = sum(r['match'] for r in results)

    fig = plt.figure(figsize=(20, 8), facecolor='#0d1117')
    outer = gridspec.GridSpec(1, 2, figure=fig, width_ratios=[1, 4], wspace=0.05)

    # ── LEFT: Query panel ────────────────────────────────────────────────────
    left_gs = gridspec.GridSpecFromSubplotSpec(3, 1, subplot_spec=outer[0],
                                               height_ratios=[3,1,1], hspace=0.05)
    ax_query = fig.add_subplot(left_gs[0])
    make_person_placeholder(ax_query, color=NEON_BLUE, is_query=True)

    ax_meta = fig.add_subplot(left_gs[1])
    ax_meta.set_facecolor('#0d1117')
    ax_meta.axis('off')
    ax_meta.text(0.5, 0.8, f'ID #{query_id:04d}', ha='center', va='top',
                 color='#ffd700', fontsize=11, fontweight='bold',
                 transform=ax_meta.transAxes)
    ax_meta.text(0.5, 0.45, 'QUERY IMAGE', ha='center', va='top',
                 color='#8b949e', fontsize=8, transform=ax_meta.transAxes)

    ax_stats = fig.add_subplot(left_gs[2])
    ax_stats.set_facecolor('#161b22')
    ax_stats.axis('off')
    ax_stats.text(0.5, 0.85, f'Rank-1: {rank1:.1f}%', ha='center', va='top',
                  color=NEON_GREEN, fontsize=10, fontweight='bold',
                  transform=ax_stats.transAxes)
    ax_stats.text(0.5, 0.55, f'mAP:    {map_score:.1f}%', ha='center', va='top',
                  color=NEON_PURPLE, fontsize=10, fontweight='bold',
                  transform=ax_stats.transAxes)
    ax_stats.text(0.5, 0.25, f'Iter:  {current_iter:>5d}', ha='center', va='top',
                  color='#8b949e', fontsize=8, transform=ax_stats.transAxes)

    # ── RIGHT: Gallery rank list ──────────────────────────────────────────────
    right_gs = gridspec.GridSpecFromSubplotSpec(2, TOP_K, subplot_spec=outer[1],
                                                height_ratios=[4,1], hspace=0.05, wspace=0.08)

    for k, res in enumerate(results):
        ax_g = fig.add_subplot(right_gs[0, k])
        color = NEON_GREEN if res['match'] else NEON_ORANGE
        make_person_placeholder(ax_g, color=color)

        # Rank border
        border_color = '#2ea043' if res['match'] else '#da3633'
        for spine in ax_g.spines.values():
            spine.set_edgecolor(border_color)
            spine.set_linewidth(2)

        ax_s = fig.add_subplot(right_gs[1, k])
        ax_s.set_facecolor('#0d1117')
        ax_s.axis('off')

        # Score bar
        bar_color = NEON_GREEN if res['match'] else NEON_ORANGE
        ax_s.barh(0, res['score'], color=bar_color, height=0.5, alpha=0.8)
        ax_s.barh(0, 1.0, color='#21262d', height=0.5, alpha=0.6)
        ax_s.barh(0, res['score'], color=bar_color, height=0.5, alpha=0.9)
        ax_s.set_xlim(0,1)
        ax_s.set_ylim(-0.5, 1)

        label = '✓ MATCH' if res['match'] else '✗ WRONG'
        ax_s.text(0.5, 0.85, f'#{k+1}  {res["score"]:.2f}',
                  ha='center', va='top', color='white', fontsize=7.5,
                  fontweight='bold', transform=ax_s.transAxes)
        ax_s.text(0.5, 0.35, label, ha='center', va='top',
                  color=bar_color, fontsize=6.5, transform=ax_s.transAxes)

    # ── TITLE BAR ─────────────────────────────────────────────────────────────
    status_color = NEON_GREEN if n_correct >= TOP_K * 0.7 else NEON_ORANGE
    status_text  = f'Re-ID: {n_correct}/{TOP_K} Correct  ({n_correct/TOP_K*100:.0f}%)'

    fig.text(0.5, 0.98, status_text,
             ha='center', va='top', fontsize=15, color=status_color, fontweight='bold')
    fig.text(0.5, 0.93, f'FastReID — Market1501 Validation  |  Iter {current_iter}',
             ha='center', va='top', fontsize=10, color='#8b949e')

    fig.text(0.01, 0.98, '●', color=NEON_GREEN, fontsize=12, va='top')
    fig.text(0.03, 0.98, '= Correct match', color='#8b949e', fontsize=9, va='top')
    fig.text(0.01, 0.93, '●', color=NEON_ORANGE, fontsize=12, va='top')
    fig.text(0.03, 0.93, '= Wrong match', color='#8b949e', fontsize=9, va='top')

    plt.savefig(f'../outputs/reid_validation_q{query_id:04d}.png', dpi=140,
                bbox_inches='tight', facecolor='#0d1117')
    plt.show()
    print(f'✅ Saved → ../outputs/reid_validation_q{query_id:04d}.png')


# Show validation panel for a few sample queries
for qid in [1, 42, 100]:
    reid_validation_panel(query_id=qid, current_iter=18000, rank1=96.3, map_score=90.3)

## 🔬 Step 7 — Test & Evaluate

In [ ]:
# ─── RUN EVALUATION ───────────────────────────────────────────────────────────
# After training completes, run evaluation with different post-processing

OUTPUT_DIR  = '../outputs/market1501_best'
MODEL_PATH  = f'{OUTPUT_DIR}/model_final.pth'

# ── Baseline eval (no post-processing) ────────────────────────────────────────
eval_cmd = (
    f'python tools/train_net.py '
    f'--config-file configs/Market1501/best_config.yml '
    f'--eval-only '
    f'MODEL.WEIGHTS {MODEL_PATH} '
    f'TEST.RERANK.ENABLED False '
    f'2>&1 | tee ../outputs/eval_baseline.txt'
)

# ── With Query Expansion ───────────────────────────────────────────────────────
eval_qe_cmd = (
    f'python tools/train_net.py '
    f'--config-file configs/Market1501/best_config.yml '
    f'--eval-only '
    f'MODEL.WEIGHTS {MODEL_PATH} '
    f'TEST.AQE.ENABLED True '
    f'2>&1 | tee ../outputs/eval_qe.txt'
)

# ── With Re-ranking ────────────────────────────────────────────────────────────
eval_rerank_cmd = (
    f'python tools/train_net.py '
    f'--config-file configs/Market1501/best_config.yml '
    f'--eval-only '
    f'MODEL.WEIGHTS {MODEL_PATH} '
    f'TEST.RERANK.ENABLED True '
    f'2>&1 | tee ../outputs/eval_rerank.txt'
)

print('Eval commands ready. Uncomment to run after training:')
print(f'  Baseline : {eval_cmd[:80]}...')
print(f'  + QE     : {eval_qe_cmd[:80]}...')
print(f'  + Rerank : {eval_rerank_cmd[:80]}...')

# Uncomment ONE at a time after training:
# !{eval_cmd}
# !{eval_qe_cmd}
# !{eval_rerank_cmd}

In [ ]:
# ─── FINAL RESULTS TABLE ──────────────────────────────────────────────────────
# Populate these after running the eval commands above

import pandas as pd

# From paper Table 1 — FastReID on Market1501
results_df = pd.DataFrame([
    {'Config': 'FastReID (ResNet50)',      'Rank-1': 95.4, 'mAP': 88.2, 'Post-proc': 'None'},
    {'Config': 'FastReID (ResNet50-IBN)',  'Rank-1': 95.7, 'mAP': 89.3, 'Post-proc': 'None'},
    {'Config': 'FastReID (ResNeSt)',       'Rank-1': 95.0, 'mAP': 87.0, 'Post-proc': 'None'},
    {'Config': 'FastReID-MGN (R50-IBN)',   'Rank-1': 95.7, 'mAP': 89.7, 'Post-proc': 'None'},
    {'Config': 'FastReID (ResNet101-IBN)', 'Rank-1': 96.3, 'mAP': 90.3, 'Post-proc': 'None'},
    {'Config': 'FastReID (R101-IBN) + QE', 'Rank-1': 96.5, 'mAP': 94.4, 'Post-proc': 'Query Exp.'},
    {'Config': 'FastReID (R101-IBN) + RR', 'Rank-1': 96.8, 'mAP': 95.3, 'Post-proc': 'Re-rank'},
])

print('\n📋 FastReID Market1501 Results (from paper Table 1):')
print('=' * 65)
print(results_df.to_string(index=False))
print('\n🏆 Best: Rank-1=96.8%, mAP=95.3% (with Re-ranking)')

In [ ]:
# ─── POST-PROCESSING COMPARISON PLOT ─────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5), facecolor='#0d1117')

configs  = ['R50', 'R50-IBN', 'ResNeSt', 'MGN\n(R50-IBN)', 'R101-IBN',
            'R101-IBN\n+QE', 'R101-IBN\n+ReRank']
r1_vals  = [95.4, 95.7, 95.0, 95.7, 96.3, 96.5, 96.8]
map_vals = [88.2, 89.3, 87.0, 89.7, 90.3, 94.4, 95.3]
colors   = [NEON_BLUE]*5 + [NEON_YELLOW, NEON_GREEN]

for ax, vals, title, target in [
    (ax1, r1_vals,  'Rank-1 Accuracy', 96.3),
    (ax2, map_vals, 'mAP Score',       90.3),
]:
    ax.set_facecolor('#161b22')
    bars = ax.bar(range(len(configs)), vals, color=colors, alpha=0.88, width=0.65)
    ax.axhline(target, color='white', ls='--', lw=1, alpha=0.4, label=f'Baseline target {target}%')
    for i, (b, v) in enumerate(zip(bars, vals)):
        ax.text(b.get_x() + b.get_width()/2, v + 0.1, f'{v:.1f}%',
                ha='center', va='bottom', color='white', fontsize=8)
    ax.set_xticks(range(len(configs)))
    ax.set_xticklabels(configs, fontsize=8, color='#8b949e')
    ax.set_ylim(min(vals)-2, max(vals)+2)
    ax.set_title(title, color='white', fontweight='bold')
    ax.grid(True, alpha=0.15, axis='y')
    ax.spines[['top','right']].set_visible(False)
    for s in ax.spines.values():
        s.set_edgecolor('#30363d')
    ax.tick_params(colors='#8b949e')

fig.suptitle('FastReID Market1501 — Config & Post-processing Comparison', 
             color='white', fontsize=13, fontweight='bold')

legend_patches = [
    mpatches.Patch(color=NEON_BLUE,   label='Base configs'),
    mpatches.Patch(color=NEON_YELLOW, label='+ Query Expansion'),
    mpatches.Patch(color=NEON_GREEN,  label='+ Re-ranking'),
]
fig.legend(handles=legend_patches, loc='lower center', ncol=3,
           facecolor='#21262d', edgecolor='#30363d', labelcolor='white', fontsize=9)

plt.savefig('../outputs/model_comparison.png', dpi=140, bbox_inches='tight',
            facecolor='#0d1117')
plt.show()
print('✅ Saved → ../outputs/model_comparison.png')

## 🚢 Step 8 — Quick-Start Commands Reference

In [ ]:
print("""
╔══════════════════════════════════════════════════════════════════╗
║          FastReID Market1501 — Quick Reference                   ║
╠══════════════════════════════════════════════════════════════════╣
║  TRAIN (single GPU):                                            ║
║    python tools/train_net.py \\                                  ║
║      --config-file configs/Market1501/best_config.yml           ║
║                                                                  ║
║  TRAIN (multi-GPU, e.g. 4):                                      ║
║    python tools/train_net.py \\                                  ║
║      --config-file configs/Market1501/best_config.yml \\         ║
║      --num-gpus 4                                                ║
║                                                                  ║
║  EVAL (baseline):                                                ║
║    python tools/train_net.py --eval-only \\                      ║
║      --config-file configs/Market1501/best_config.yml \\         ║
║      MODEL.WEIGHTS outputs/.../model_final.pth                  ║
║                                                                  ║
║  EVAL (+ Re-ranking):                                            ║
║    ...same + TEST.RERANK.ENABLED True                            ║
║                                                                  ║
║  TENSORBOARD:                                                    ║
║    tensorboard --logdir outputs/market1501_best                  ║
╠══════════════════════════════════════════════════════════════════╣
║  TARGET RESULTS (ResNet101-IBN, from paper):                    ║
║    Rank-1 = 96.3%   mAP = 90.3%  (baseline)                    ║
║    Rank-1 = 96.5%   mAP = 94.4%  (+ Query Expansion)           ║
║    Rank-1 = 96.8%   mAP = 95.3%  (+ Re-ranking)                ║
╚══════════════════════════════════════════════════════════════════╝
""")